# Notebook 05 — Baseline Models

**PharmaLens: An Explainable AI Framework for Drug–Target Interaction Prediction**

---

## Objective

Establish **baseline performance** using simple models before building complex architectures. Any useful model should significantly outperform these baselines.

We train and evaluate:
1. **Mean Predictor** — Always predicts the training set mean (absolute lower bound)
2. **Ridge Regression** — Linear model with L2 regularization
3. **Random Forest** — Ensemble of decision trees

All models use the same input: concatenated drug features (descriptors + Morgan FP) + protein features (AAC + DPC).

In [1]:
import os, sys, warnings, pickle, time
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.utils.config import (
    PROCESSED_DATA_DIR, SPLITS_DIR, FIGURES_DIR, TABLES_DIR,
    ensure_dirs, set_seed, set_plot_style, MODEL_COLORS
)
from src.evaluation.metrics import compute_regression_metrics, format_metrics_table
from src.evaluation.experiment_tracker import ExperimentTracker
from src.models.baselines import (
    MeanPredictor, get_baseline_models,
    train_and_evaluate_baseline, save_baseline_model
)

set_seed(42)
COLORS = set_plot_style()
ensure_dirs()
tracker = ExperimentTracker()
print("Setup complete ✓")

Setup complete ✓


## 1. Prepare Feature Matrices

In [2]:
# Load pre-computed features
with open(PROCESSED_DATA_DIR / 'drug_features.pkl', 'rb') as f:
    drug_features = pickle.load(f)
with open(PROCESSED_DATA_DIR / 'protein_features.pkl', 'rb') as f:
    protein_features = pickle.load(f)

print(f"Drug features loaded: {len(drug_features)} drugs")
print(f"Protein features loaded: {len(protein_features)} targets")

# Load splits
train_df = pd.read_csv(SPLITS_DIR / 'random_train.csv')
val_df = pd.read_csv(SPLITS_DIR / 'random_val.csv')
test_df = pd.read_csv(SPLITS_DIR / 'random_test.csv')

print(f"\nTrain: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")

Drug features loaded: 2111 drugs
Protein features loaded: 229 targets

Train: 94,602 | Val: 11,826 | Test: 11,826


In [3]:
def build_feature_matrix(df, drug_features, protein_features):
    """Build concatenated drug+protein feature matrix and target vector."""
    X_list, y_list = [], []
    skipped = 0
    
    for _, row in df.iterrows():
        drug_id = row['drug_id']
        target_id = row['target_id']
        
        if drug_id in drug_features and target_id in protein_features:
            drug_fv = drug_features[drug_id]
            prot_fv = protein_features[target_id]
            combined = np.concatenate([drug_fv, prot_fv])
            X_list.append(combined)
            y_list.append(row['score'])
        else:
            skipped += 1
    
    if skipped > 0:
        print(f"  Skipped {skipped} rows (missing features)")
    
    return np.array(X_list, dtype=np.float32), np.array(y_list, dtype=np.float32)

print("Building feature matrices...")
X_train, y_train = build_feature_matrix(train_df, drug_features, protein_features)
X_val, y_val = build_feature_matrix(val_df, drug_features, protein_features)
X_test, y_test = build_feature_matrix(test_df, drug_features, protein_features)

print(f"\nX_train: {X_train.shape}  y_train: {y_train.shape}")
print(f"X_val:   {X_val.shape}  y_val:   {y_val.shape}")
print(f"X_test:  {X_test.shape}  y_test:  {y_test.shape}")
print(f"Feature dimension: {X_train.shape[1]}")

Building feature matrices...

X_train: (94602, 1454)  y_train: (94602,)
X_val:   (11826, 1454)  y_val:   (11826,)
X_test:  (11826, 1454)  y_test:  (11826,)
Feature dimension: 1454


## 2. Train Baseline Models

In [ ]:
# Train all baselines
models = get_baseline_models()
all_results = {}

for name, model in models.items():
    print(f"\n{'='*60}")
    results = train_and_evaluate_baseline(
        model, name, X_train, y_train, X_val, y_val, X_test, y_test
    )
    all_results[name] = results
    
    # Log to experiment tracker
    tracker.log_experiment(
        model=name,
        dataset='KIBA',
        representation='descriptors+fingerprints+aac+dpc',
        split='random',
        seed=42,
        metrics=results['test_metrics'],
        training_time=results['training_time'],
        inference_time=results['inference_time'],
    )
    
    # Save model
    save_baseline_model(model, name)

print(f"\n{'='*60}")
print("All baselines trained ✓")


✓ Logged experiment: EXP_20260930_190700_827043 | MeanPredictor | random | seed=42

✓ Logged experiment: EXP_20260930_190702_926c53 | Ridge | random | seed=42



## 3. Results Comparison

In [ ]:
# Build comparison table
comparison_rows = []
for name, results in all_results.items():
    row = {'Model': name}
    row.update({f'Test {k.upper()}': v for k, v in results['test_metrics'].items() 
                if k in ['rmse', 'mae', 'pearson', 'spearman', 'r2', 'ci']})
    row['Train Time (s)'] = results['training_time']
    comparison_rows.append(row)

comparison_df = pd.DataFrame(comparison_rows)
print("\nBaseline Model Comparison (Test Set):")
print(comparison_df.to_string(index=False))

# Save
comparison_df.to_csv(TABLES_DIR / 'baseline_comparison.csv', index=False)

In [ ]:
# Visualize: Actual vs Predicted for each model
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, (name, results) in enumerate(all_results.items()):
    y_pred = results['predictions']['y_test_pred']
    color = MODEL_COLORS.get(name, COLORS['primary'])
    
    axes[idx].scatter(y_test[:len(y_pred)], y_pred, alpha=0.15, s=5, color=color)
    lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
    axes[idx].plot(lims, lims, 'k--', alpha=0.5, linewidth=1)
    axes[idx].set_xlabel('Actual KIBA Score')
    axes[idx].set_ylabel('Predicted KIBA Score')
    rmse = results['test_metrics']['rmse']
    pearson = results['test_metrics']['pearson']
    axes[idx].set_title(f'{name}\nRMSE={rmse:.3f}, r={pearson:.3f}', fontweight='bold')

plt.suptitle('Actual vs Predicted — Baseline Models', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'baseline_actual_vs_predicted.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Residual plots
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, (name, results) in enumerate(all_results.items()):
    y_pred = results['predictions']['y_test_pred']
    residuals = y_test[:len(y_pred)] - y_pred
    color = MODEL_COLORS.get(name, COLORS['primary'])
    
    axes[idx].scatter(y_pred, residuals, alpha=0.15, s=5, color=color)
    axes[idx].axhline(y=0, color='k', linestyle='--', alpha=0.5)
    axes[idx].set_xlabel('Predicted KIBA Score')
    axes[idx].set_ylabel('Residual')
    axes[idx].set_title(f'{name} — Residuals', fontweight='bold')

plt.suptitle('Residual Plots — Baseline Models', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'baseline_residuals.png', dpi=300, bbox_inches='tight')
plt.show()

## Observations

1. **Mean Predictor** serves as the absolute lower bound. Any model with lower RMSE is learning something beyond the training set average.

2. **Ridge Regression** captures linear relationships between features and interaction scores.

3. **Random Forest** typically outperforms linear models due to its ability to capture nonlinear patterns and feature interactions.

4. These baselines establish the performance bar that XGBoost, DeepDTA, and GNN models must exceed to justify their additional complexity.

## Conclusion

Baseline performance is established. All results are logged to the experiment tracker for later comparison.

**Next:** [Notebook 06 — XGBoost Model](./06_xgboost_model.ipynb)